## In-situ Dataset
In this notebook, we find the time sereis associated with POT variable and the associated variables' time sereis. 

In [1]:

import os
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
locations = ['OceanSprings']
properties = ['TimeVarying','Stationary']
target_events = 5

core_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/Datasets')

time_rf = 'time'
time_ntr = 'time'
time_rd = 'dateTime'

time_pot_rf = 'time'
time_pot_ntr = 'time'
time_pot_rd = 'time'

ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

In [ ]:

for prop in properties:
    for loc in locations:
        base_path = core_directory / loc / prop
        
        df_pot_ntr = pd.read_csv(rf'{base_path}/POT_NTR_{prop}_RF_RD_{target_events}.csv')
        df_pot_rf = pd.read_csv(rf'{base_path}/POT_RF_{prop}_NTR_RD_{target_events}.csv')
        df_pot_rd = pd.read_csv(rf'{base_path}/POT_RD_{prop}_NTR_RF_{target_events}.csv')

        df_pot_rd['time'] = (
            pd.to_datetime(
                df_pot_rd[time_pot_rd].astype(str).str.strip(),
                errors='coerce',
                infer_datetime_format=True
            )
            .dt.strftime('%Y-%m-%d %H:%M:%S')
        )
        
        df_timeseries_ntr = pd.read_csv(rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv')
        df_timeseries_ntr = df_timeseries_ntr.rename(columns={time_ntr:'time'})
        df_timeseries_ntr = df_timeseries_ntr[['time', ntr_col]]
        
        df_timeseries_rf = pd.read_csv(rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv')
        df_timeseries_rf = df_timeseries_rf.rename(columns={time_rf:'time'})
        df_timeseries_rf = df_timeseries_rf[['time', acc_hr]]
        
        df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/In_Situ/usgs_discharge.csv')
        df_timeseries_rd = df_timeseries_rd.rename(columns={time_rd:'time'})
        df_timeseries_rd = df_timeseries_rd[['time', rd_col]]
        
        # as pandas datetime (1952-10-01 00:00:00 etc.)
        df_timeseries_rd['time'] = pd.to_datetime(
            df_timeseries_rd['time'].astype(str).str.strip(),
            format='%m/%d/%Y',   
            errors='coerce'
        )
        
        df_timeseries_rd = df_timeseries_rd.copy()
        df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.strftime('%Y-%m-%d %H:%M:%S')     
        
        df_pot_rf['time']          = pd.to_datetime(df_pot_rf[time_pot_rf], errors='coerce')
        df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr['time'], errors='coerce')
        df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf['time'],  errors='coerce')
        df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd['time'],  errors='coerce')
        
        # --- Build an HOURLY base from RF ⨉ NTR ---
        hourly_base = (
            df_timeseries_rf
              .merge(df_timeseries_ntr, on='time', how='inner')  
              .sort_values('time')
        )
        hourly_base['date'] = hourly_base['time'].dt.normalize() 
        
        # --- Map the DAILY RD value to each hour by DATE  ---
        df_rd_daily = df_timeseries_rd[['time', rd_col]].copy()
        df_rd_daily['date'] = df_rd_daily['time'].dt.normalize()
        df_rd_daily = df_rd_daily[['date', rd_col]]
        
        hourly_base = hourly_base.merge(df_rd_daily, on='date', how='left')
        
        # Now hourly_base has columns: ['time', acc_hr, ntr_col, 'date', rd_col]
        
        # --- Build event windows around POT_RF times; keep HOURLY timestamps in the output ---
        rf_windows = []
        
        for _, row in df_pot_rf.iterrows():
            pot_time = row['time']
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            # hourly window (RF+NTR, with RD value repeated for each hour of the day)
            window_df = hourly_base.loc[
                hourly_base['time'].between(window_start, window_end)
            ].copy()
            window_df['event_time'] = pot_time
        
            # --- max NTR within ±5 days using HOURLY NTR ---
            sub_ntr = window_df.loc[
                window_df['time'].between(pot_time - pd.Timedelta(days=5),
                                          pot_time + pd.Timedelta(days=5)),
                ['time', ntr_col]
            ].dropna()
        
            if not sub_ntr.empty:
                i = sub_ntr[ntr_col].idxmax()
                max_ntr_time = sub_ntr.at[i, 'time']
                window_df['ntr_lag_hours'] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['ntr_lag_hours'] = np.nan
        
            # --- max RD within ±5 days using the ORIGINAL DAILY RD series ---
            sub_rd = df_timeseries_rd.loc[
                df_timeseries_rd['time'].between(pot_time - pd.Timedelta(days=5),
                                                 pot_time + pd.Timedelta(days=5)),
                ['time', rd_col]
            ].dropna()
        
            if not sub_rd.empty:
                j = sub_rd[rd_col].idxmax()
                max_rd_time = sub_rd.at[j, 'time']        # daily timestamp (likely 00:00:00)
                window_df['rd_lag_hours'] = (max_rd_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rd_lag_hours'] = np.nan
        
            rf_windows.append(window_df)
        
        df_windows_total = pd.concat(rf_windows, ignore_index=True)
        
        # Keep datetime dtype; only format on CSV write if needed:
        out_path = rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv'
        df_windows_total.to_csv(out_path, index=False, date_format='%Y-%m-%d %H:%M:%S')
        
        ########################################################
        ########################################################
        #NTR
        # --- POT NTR times as datetime ---
        df_pot_ntr['time'] = pd.to_datetime(df_pot_ntr.pop(time_pot_ntr), errors='coerce')
        
        # --- time series as datetime (keep as datetime; do NOT strftime) ---
        df_timeseries_ntr['time'] = pd.to_datetime(df_timeseries_ntr[time_ntr], errors='coerce')
        df_timeseries_rf['time']  = pd.to_datetime(df_timeseries_rf[time_rf],  errors='coerce')
        
        # RD may be mm/dd/YYYY or already ISO; try specific then fallback
        try:
            df_timeseries_rd['time'] = pd.to_datetime(
                df_timeseries_rd[time_rd].astype(str).str.strip(),
                format='%m/%d/%Y', errors='raise'
            )
        except Exception:
            df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd[time_rd], errors='coerce')
        
        # ---------------------------
        # Build HOURLY base (RF × NTR)
        # ---------------------------
        hourly_base = (
            df_timeseries_ntr[['time', ntr_col]]
              .merge(df_timeseries_rf[['time', acc_hr]], on='time', how='inner')
              .sort_values('time')
        )
        hourly_base['date'] = hourly_base['time'].dt.normalize()  # YYYY-MM-DD 00:00:00
        
        # Attach DAILY RD by date (no resampling/interpolation of values)
        rd_daily = df_timeseries_rd[['time', rd_col]].copy()
        rd_daily['date'] = rd_daily['time'].dt.normalize()
        rd_daily = rd_daily[['date', rd_col]]
        
        hourly_base = hourly_base.merge(rd_daily, on='date', how='left')
        
        # ---------------------------
        # Windows around each POT NTR
        # ---------------------------
        rf_windows = []
        
        for _, row in df_pot_ntr.iterrows():
            pot_time = row['time']
            if pd.isna(pot_time):
                continue
        
            # ±30-day hourly window (keeps hourly timestamps)
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            window_df = hourly_base.loc[
                hourly_base['time'].between(window_start, window_end)
            ].copy()
            window_df['event_time'] = pot_time
        
            # ---- RF max within ±5 days (hourly)
            sub_rf = window_df.loc[
                window_df['time'].between(pot_time - pd.Timedelta(days=5),
                                          pot_time + pd.Timedelta(days=5)),
                ['time', acc_hr]
            ].dropna(subset=[acc_hr])
        
            if not sub_rf.empty:
                i = sub_rf[acc_hr].idxmax()
                max_rf_time = sub_rf.at[i, 'time']
                window_df['rf_lag_hours'] = (max_rf_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rf_lag_hours'] = np.nan
        
            # ---- RD max within ±5 days (use ORIGINAL DAILY series)
            sub_rd = df_timeseries_rd.loc[
                df_timeseries_rd['time'].between(pot_time - pd.Timedelta(days=5),
                                                 pot_time + pd.Timedelta(days=5)),
                ['time', rd_col]
            ].dropna(subset=[rd_col])
        
            if not sub_rd.empty:
                j = sub_rd[rd_col].idxmax()
                max_rd_time = sub_rd.at[j, 'time']  # daily timestamp (00:00:00)
                window_df['rd_lag_hours'] = (max_rd_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rd_lag_hours'] = np.nan
        
            rf_windows.append(window_df)
        
        # Combine all windows; keep datetime dtype (hours preserved)
        df_windows_total = pd.concat(rf_windows, ignore_index=True)
        df_windows_total = df_windows_total.sort_values(['event_time', 'time']).reset_index(drop=True)
        
        # Save; date_format controls how timestamps appear in the CSV text, dtype stays datetime in-memory
        out_csv = rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv'
        df_windows_total.to_csv(out_csv, index=False, date_format='%Y-%m-%d %H:%M:%S')
        print(f"saved → {out_csv}")
        
        ################################################################################
        #################################################################################
        #RD
        # --- Parse datetimes (keep as datetime64[ns], do NOT strftime) ---
        df_pot_rf['time']          = pd.to_datetime(df_pot_rf[time_pot_rf], errors='coerce')
        df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr[time_ntr], errors='coerce')
        df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf[time_rf],  errors='coerce')
        df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd[time_rd],  errors='coerce')
        
        # --- Make sure all 'time' columns are datetime ---
        # POT RD triggers
        if 'Time' in df_pot_rd.columns and 'time' not in df_pot_rd.columns:
            df_pot_rd = df_pot_rd.rename(columns={'Time': 'time'})
        
        df_pot_rd['time'] = pd.to_datetime(df_pot_rd['time'], errors='coerce')      
        
        def expand_daily_to_hourly(df_daily, time_col='time', value_col=rd_col):
            """
            For each daily timestamp in df_daily[time_col], create 24 hourly rows from
            00:00 to 23:00 with the same value in value_col.
            """
            df_daily = df_daily.dropna(subset=[time_col, value_col]).copy()
            df_daily[time_col] = pd.to_datetime(df_daily[time_col], errors='coerce')
            df_daily[value_col] = pd.to_numeric(df_daily[value_col], errors='coerce')
            df_daily = df_daily.dropna(subset=[time_col, value_col])
        
            pieces = []
            for t, v in zip(df_daily[time_col], df_daily[value_col]):
                d = pd.Timestamp(t).normalize()  # midnight of that day
                hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
                pieces.append(pd.DataFrame({time_col: hours, value_col: v}))
            if not pieces:
                return df_daily.iloc[0:0][[time_col, value_col]]  # empty with same cols
            return pd.concat(pieces, ignore_index=True)
        
        
        
        # Optional: drop rows with bad times
        df_pot_rd = df_pot_rd.dropna(subset=['time'])
        
        rd_windows = []
        
        for _, row in df_pot_rd.iterrows():
            pot_time = row['time']
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            window_df_rd_daily = df_timeseries_rd.loc[
                (df_timeseries_rd['time'] >= window_start) &
                (df_timeseries_rd['time'] <= window_end)
            ].copy()
        
            # ⬇️ expand daily RD to hourly within the window
            window_df_rd = expand_daily_to_hourly(
                window_df_rd_daily, time_col='time', value_col=rd_col
            )
        
            # keep NTR/RF hourly (unchanged)
            window_df_ntr = df_timeseries_ntr.loc[
                (df_timeseries_ntr['time'] >= window_start) &
                (df_timeseries_ntr['time'] <= window_end)
            ].copy()
        
            window_df_rf = df_timeseries_rf.loc[
                (df_timeseries_rf['time'] >= window_start) &
                (df_timeseries_rf['time'] <= window_end)
            ].copy()
        
            # inner-join on hourly timestamps; RD is now hourly by repetition
            window_df = (
                window_df_rd
                .merge(window_df_ntr, on='time', how='inner')
                .merge(window_df_rf, on='time', how='inner')
                .sort_values('time')
                .reset_index(drop=True)
            )
            window_df['event_time'] = pot_time
        
            if not window_df.empty:
                window_start = pot_time - pd.Timedelta(days=5)
                window_end   = pot_time + pd.Timedelta(days=5)
                sub = window_df.loc[
                window_df['time'].between(window_start, window_end),
                    ['time', acc_hr]
                ].dropna(subset=[acc_hr])
                
                if not sub.empty:
                    i = sub[acc_hr].idxmax()
                    max_rf_time = sub.at[i, 'time']
                    max_rf_val  = sub.at[i, acc_hr]
                    window_df['rf_lag_hours'] = (max_rf_time - pot_time) / pd.Timedelta(hours=1)
            
                sub = window_df.loc[
                window_df['time'].between(window_start, window_end),
                    ['time', ntr_col]
                ].dropna(subset=[ntr_col])
                
                if not sub.empty:
                    i = sub[ntr_col].idxmax()
                    max_ntr_time = sub.at[i, 'time']
                    max_ntr_val  = sub.at[i, ntr_col]
                    window_df['ntr_lag_hours'] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
                    
                else:
                    window_df['rf_lag_hours'] = np.nan
                    window_df['ntr_lag_hours']  = np.nan
        
            rd_windows.append(window_df)
        
        df_windows_total = pd.concat(rd_windows, ignore_index=True)
        
        out_path = rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv'
        df_windows_total.to_csv(out_path, index=False)
        print(f"saved → {out_path}")


C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:15: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(


saved → C:\Users\sadaf\Desktop\Generating_Boundary_Conditions\POT_24hr_Jan26\Trivariate\CorpusChristi\TimeVarying/TimeSeries_POT_NTR_Associated_RF_RD.csv


C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning

saved → C:\Users\sadaf\Desktop\Generating_Boundary_Conditions\POT_24hr_Jan26\Trivariate\CorpusChristi\TimeVarying/TimeSeries_POT_RD_Associated_NTR_RF.csv


C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:15: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(


...

saved → C:\Users\sadaf\Desktop\Generating_Boundary_Conditions\POT_24hr_Jan26\Trivariate\CorpusChristi\Stationary/TimeSeries_POT_RD_Associated_NTR_RF.csv


C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:15: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(


saved → C:\Users\sadaf\Desktop\Generating_Boundary_Conditions\POT_24hr_Jan26\Trivariate\OceanSprings\Stationary/TimeSeries_POT_NTR_Associated_RF_RD.csv


C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\76173049.py:253: FutureWarning

saved → C:\Users\sadaf\Desktop\Generating_Boundary_Conditions\POT_24hr_Jan26\Trivariate\OceanSprings\Stationary/TimeSeries_POT_RD_Associated_NTR_RF.csv


The generated river discharge (RD) time series was based on the USGS station with the longest available record (extending back to 1950). Using this generated series as a reference, we derived the time series for all upstream rivers. Specifically, for Ocean Springs.

## Ocean Springs

In [ ]:
properties = ['TimeVarying','Stationary']
target_events = 5
    
core_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/POT_24hr_Jan26/Trivariate/OceanSprings')
dataset_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/Datasets/Observed_OceanSprings')

time_rf = 'time'
time_ntr = 'time'
time_rd = 'datetime'

time_pot_rf = 'time'
time_pot_ntr = 'time'
time_pot_rd = 'time'

acc_hr = 'acc_24hr'
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
time_rd_2 = 'Unnamed: 0' #if your datasets for RD has two different names for time column

In [ ]:
for prop in properties:
    
    base_path = core_directory / prop

    df_pot_ntr = pd.read_csv(rf'{base_path}/POT_NTR_{prop}_RF_RD_NOAA_GHCN_USGS_{target_events}.csv')
    df_pot_rf = pd.read_csv(rf'{base_path}/POT_RF_{prop}_NTR_RD_GHCN_NOAA_USGS_{target_events}.csv')
    df_pot_rd = pd.read_csv(rf'{base_path}/POT_RD_{prop}_NTR_RF_USGS_NOAA_GHCN_{target_events}.csv')

    df_pot_rd['time'] = (
        pd.to_datetime(
            df_pot_rd[time_pot_rd].astype(str).str.strip(),
            errors='coerce',
            infer_datetime_format=True
        )
        .dt.strftime('%Y-%m-%d %H:%M:%S')
    )

    df_timeseries_ntr = pd.read_csv(rf'{dataset_directory}/NOAA_Tide_Gauge_MSL_All_Predicted.csv')
    df_timeseries_ntr = df_timeseries_ntr[[time_ntr, ntr_col]]
    
    df_timeseries_rf = pd.read_csv(rf'{dataset_directory}/GHCN_Rainfall_1950_2050_hourly_accumulated.csv')
    df_timeseries_rf = df_timeseries_rf[[time_rf, acc_hr]]
    
    df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/usgs_02479310_discharge_hourly.csv')
    df_timeseries_rd = df_timeseries_rd.rename(columns={time_rd_2:'time'})
    df_timeseries_rd = df_timeseries_rd[['time',rd_col]]
    df_timeseries_rd = df_timeseries_rd.rename(columns={rd_col:'discharge_m3s_hourly'})
    df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd['time'])        # ensure it's datetime
    df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.tz_localize(None)   # strip timezone info
    
    df_pot_ntr_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv')
    df_pot_rf_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv')
    df_pot_rd_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv')
    df_pot_ntr_timeseries['time']  = pd.to_datetime(df_pot_ntr_timeseries[time_pot_ntr], errors='coerce')
    df_pot_rf_timeseries['time']   = pd.to_datetime(df_pot_rf_timeseries[time_pot_rf],  errors='coerce')
    df_pot_rd_timeseries['time']   = pd.to_datetime(df_pot_rd_timeseries[time_pot_rd],  errors='coerce')
    
    df_pot_ntr_timeseries_merged = pd.merge(df_pot_ntr_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    
    rd_start = df_timeseries_rd['time'].min()
    mask = (df_pot_ntr_timeseries_merged['time'] < rd_start) | (
        (df_pot_ntr_timeseries_merged['time'] >= rd_start) &
        df_pot_ntr_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_ntr_timeseries_merged = df_pot_ntr_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_ntr_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD_Pascagoula.csv', index=False)
    
    df_pot_rf_timeseries_merged = pd.merge(df_pot_rf_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rf_timeseries_merged['time'] < rd_start) | (
        (df_pot_rf_timeseries_merged['time'] >= rd_start) &
        df_pot_rf_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rf_timeseries_merged = df_pot_rf_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rf_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD_Pascagoula.csv', index=False)
    
    df_pot_rd_timeseries_merged = pd.merge(df_pot_rd_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rd_timeseries_merged['time'] < rd_start) | (
        (df_pot_rd_timeseries_merged['time'] >= rd_start) &
        df_pot_rd_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rd_timeseries_merged = df_pot_rd_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rd_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF_Pascagoula.csv', index=False)
    
    
    df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/usgs_02481000_discharge_hourly.csv')
    df_timeseries_rd = df_timeseries_rd.rename(columns={time_rd:'time'})
    df_timeseries_rd = df_timeseries_rd[['time', rd_col]]
    df_timeseries_rd = df_timeseries_rd.rename(columns={rd_col:'discharge_m3s_hourly'})
    df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd['time'])        # ensure it's datetime
    df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.tz_localize(None)   # strip timezone info
    
    
    df_pot_ntr_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv')
    df_pot_rf_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv')
    df_pot_rd_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv')
    df_pot_ntr_timeseries['time']  = pd.to_datetime(df_pot_ntr_timeseries[time_pot_ntr], errors='coerce')
    df_pot_rf_timeseries['time']   = pd.to_datetime(df_pot_rf_timeseries[time_pot_rf],  errors='coerce')
    df_pot_rd_timeseries['time']   = pd.to_datetime(df_pot_rd_timeseries[time_pot_rd],  errors='coerce')

    df_pot_ntr_timeseries_merged = pd.merge(df_pot_ntr_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    
    rd_start = df_timeseries_rd['time'].min()
    mask = (df_pot_ntr_timeseries_merged['time'] < rd_start) | (
        (df_pot_ntr_timeseries_merged['time'] >= rd_start) &
        df_pot_ntr_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_ntr_timeseries_merged = df_pot_ntr_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_ntr_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD_Biloxi.csv', index=False)
    
    df_pot_rf_timeseries_merged = pd.merge(df_pot_rf_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rf_timeseries_merged['time'] < rd_start) | (
        (df_pot_rf_timeseries_merged['time'] >= rd_start) &
        df_pot_rf_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rf_timeseries_merged = df_pot_rf_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rf_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD_Biloxi.csv', index=False)
    
    df_pot_rd_timeseries_merged = pd.merge(df_pot_rd_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rd_timeseries_merged['time'] < rd_start) | (
        (df_pot_rd_timeseries_merged['time'] >= rd_start) &
        df_pot_rd_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rd_timeseries_merged = df_pot_rd_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rd_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF_Biloxi.csv', index=False)
    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\4233344318.py:16: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(
C:\Users\sadaf\AppData\Local\Temp\ipykernel_32900\4233344318.py:16: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  pd.to_datetime(


## Reanalysis Dataset

In [8]:
import os
import pandas as pd
import numpy as np

In [ ]:
locations = ['OceanSprings','CorpusChristi']
properties = ['Stationary','TimeVarying']
target_events = 10

core_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/Datasets/Reanalysis')

time_ntr = 'time'
time_rf = 'time'
time_rd = 'valid_time'

time_pot_ntr = 'time'
time_pot_rf = 'time'
time_pot_rd = 'time'

ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

In [ ]:
   
for prop in properties:
    for loc in locations:
        
        base_path = core_directory / Path(f'{loc}/Reanalysis/{prop}')
        
        df_pot_ntr = pd.read_csv(rf'{base_path}/POT_NTR_{prop}_RF_RD_CDS_AORC_GloFAS_{target_events}.csv')
        df_pot_rf = pd.read_csv(rf'{base_path}/POT_RF_{prop}_NTR_RD_AORC_CDS_GloFAS_{target_events}.csv')
        df_pot_rd = pd.read_csv(rf'{base_path}/POT_RD_{prop}_NTR_RF_GloFAS_CDS_AORC_{target_events}.csv')

        df_pot_rd['time'] = (
            pd.to_datetime(
                df_pot_rd[time_pot_rd].astype(str).str.strip(),
                errors='coerce',
                infer_datetime_format=True
            )
            .dt.strftime('%Y-%m-%d %H:%M:%S')
        )
                
 
        df_timeseries_ntr = pd.read_csv(rf'{dataset_directory}/Compare_Points_{loc}_Avg.csv')
        df_timeseries_ntr = df_timeseries_ntr.rename(columns={time_ntr:'time'})
        df_timeseries_ntr = df_timeseries_ntr[['time', ntr_col]]
        
        df_timeseries_rf = pd.read_csv(rf'{dataset_directory}/APCP_Basin_Average_{loc}_Combined_accumulated.csv')
        df_timeseries_rf = df_timeseries_rf.rename(columns={time_rf:'time'})
        df_timeseries_rf = df_timeseries_rf[['time', acc_hr]]
        
        df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/GloFAS_{loc}_Basin_Avg.csv')
        df_timeseries_rd = df_timeseries_rd.rename(columns={time_rd:'time'})
        df_timeseries_rd = df_timeseries_rd[['time', rd_col]]

        # as pandas datetime (1952-10-01 00:00:00 etc.)
        df_timeseries_rd['time'] = pd.to_datetime(
            df_timeseries_rd['time'].astype(str).str.strip(),
            format='%m/%d/%Y',   # your file looks like mm/dd/YYYY
            errors='coerce'
        )
   
        df_timeseries_rd = df_timeseries_rd.copy()
        df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
       
        # --- Parse datetimes (keep as datetime64[ns], do NOT strftime) ---
        df_pot_rf['time']          = pd.to_datetime(df_pot_rf[time_pot_rf], errors='coerce')
        df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr['time'], errors='coerce')
        df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf['time'],  errors='coerce')
        df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd['time'],  errors='coerce')
        
        # --- Build an HOURLY base from RF ⨉ NTR ---
        hourly_base = (
            df_timeseries_rf
              .merge(df_timeseries_ntr, on='time', how='inner')  # keep the hourly stamps shared by both
              .sort_values('time')
        )
        hourly_base['date'] = hourly_base['time'].dt.normalize()  # YYYY-MM-DD 00:00:00
        
        # --- Map the DAILY RD value to each hour by DATE (no interpolation/resampling of values) ---
        df_rd_daily = df_timeseries_rd[['time', rd_col]].copy()
        df_rd_daily['date'] = df_rd_daily['time'].dt.normalize()
        df_rd_daily = df_rd_daily[['date', rd_col]]
        
        hourly_base = hourly_base.merge(df_rd_daily, on='date', how='left')
        
        # Now hourly_base has columns: ['time', acc_hr, 'Average', 'date', 'basin_avg_dis24_m3s']
        
        # --- Build event windows around POT_RF times; keep HOURLY timestamps in the output ---
        rf_windows = []
        
        for _, row in df_pot_rf.iterrows():
            pot_time = row['time']
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            # hourly window (RF+NTR, with RD value repeated for each hour of the day)
            window_df = hourly_base.loc[
                hourly_base['time'].between(window_start, window_end)
            ].copy()
            window_df['event_time'] = pot_time
        
            # --- max NTR within ±5 days using HOURLY NTR ---
            sub_ntr = window_df.loc[
                window_df['time'].between(pot_time - pd.Timedelta(days=5),
                                          pot_time + pd.Timedelta(days=5)),
                ['time', ntr_col]
            ].dropna()
        
            if not sub_ntr.empty:
                i = sub_ntr[ntr_col].idxmax()
                max_ntr_time = sub_ntr.at[i, 'time']
                window_df['ntr_lag_hours'] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['ntr_lag_hours'] = np.nan
        
            # --- max RD within ±5 days using the ORIGINAL DAILY RD series ---
            sub_rd = df_timeseries_rd.loc[
                df_timeseries_rd['time'].between(pot_time - pd.Timedelta(days=5),
                                                 pot_time + pd.Timedelta(days=5)),
                ['time', rd_col]
            ].dropna()
        
            if not sub_rd.empty:
                j = sub_rd[rd_col].idxmax()
                max_rd_time = sub_rd.at[j, 'time']        # daily timestamp (likely 00:00:00)
                window_df['rd_lag_hours'] = (max_rd_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rd_lag_hours'] = np.nan
        
            rf_windows.append(window_df)
        
        df_windows_total = pd.concat(rf_windows, ignore_index=True)
        
        # Keep datetime dtype; only format on CSV write if needed:
        out_path = rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv'
        df_windows_total.to_csv(out_path, index=False, date_format='%Y-%m-%d %H:%M:%S')
        
        ########################################################
        ########################################################
        #NTR
        # --- POT NTR times as datetime ---
        df_pot_ntr['time'] = pd.to_datetime(df_pot_ntr.pop(time_pot_ntr), errors='coerce')
        
        # --- time series as datetime (keep as datetime; do NOT strftime) ---
        df_timeseries_ntr['time'] = pd.to_datetime(df_timeseries_ntr[time_ntr], errors='coerce')
        df_timeseries_rf['time']  = pd.to_datetime(df_timeseries_rf[time_rf],  errors='coerce')
        
        # RD may be mm/dd/YYYY or already ISO; try specific then fallback
        try:
            df_timeseries_rd['time'] = pd.to_datetime(
                df_timeseries_rd[time_rd].astype(str).str.strip(),
                format='%m/%d/%Y', errors='raise'
            )
        except Exception:
            df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd[time_rd], errors='coerce')
        
        # ---------------------------
        # Build HOURLY base (RF × NTR)
        # ---------------------------
        hourly_base = (
            df_timeseries_ntr[['time', ntr_col]]
              .merge(df_timeseries_rf[['time', acc_hr]], on='time', how='inner')
              .sort_values('time')
        )
        hourly_base['date'] = hourly_base['time'].dt.normalize()  # YYYY-MM-DD 00:00:00
        
        # Attach DAILY RD by date (no resampling/interpolation of values)
        rd_daily = df_timeseries_rd[['time', rd_col]].copy()
        rd_daily['date'] = rd_daily['time'].dt.normalize()
        rd_daily = rd_daily[['date', rd_col]]
        
        hourly_base = hourly_base.merge(rd_daily, on='date', how='left')
        
        # ---------------------------
        # Windows around each POT NTR
        # ---------------------------
        rf_windows = []
        
        for _, row in df_pot_ntr.iterrows():
            pot_time = row['time']
            if pd.isna(pot_time):
                continue
        
            # ±30-day hourly window (keeps hourly timestamps)
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            window_df = hourly_base.loc[
                hourly_base['time'].between(window_start, window_end)
            ].copy()
            window_df['event_time'] = pot_time
        
            # ---- RF max within ±5 days (hourly)
            sub_rf = window_df.loc[
                window_df['time'].between(pot_time - pd.Timedelta(days=5),
                                          pot_time + pd.Timedelta(days=5)),
                ['time', acc_hr]
            ].dropna(subset=[acc_hr])
        
            if not sub_rf.empty:
                i = sub_rf[acc_hr].idxmax()
                max_rf_time = sub_rf.at[i, 'time']
                window_df['rf_lag_hours'] = (max_rf_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rf_lag_hours'] = np.nan
        
            # ---- RD max within ±5 days (use ORIGINAL DAILY series)
            sub_rd = df_timeseries_rd.loc[
                df_timeseries_rd['time'].between(pot_time - pd.Timedelta(days=5),
                                                 pot_time + pd.Timedelta(days=5)),
                ['time', rd_col]
            ].dropna(subset=[rd_col])
        
            if not sub_rd.empty:
                j = sub_rd[rd_col].idxmax()
                max_rd_time = sub_rd.at[j, 'time']  # daily timestamp (00:00:00)
                window_df['rd_lag_hours'] = (max_rd_time - pot_time) / pd.Timedelta(hours=1)
            else:
                window_df['rd_lag_hours'] = np.nan
        
            rf_windows.append(window_df)
        
        # Combine all windows; keep datetime dtype (hours preserved)
        df_windows_total = pd.concat(rf_windows, ignore_index=True)
        df_windows_total = df_windows_total.sort_values(['event_time', 'time']).reset_index(drop=True)
        
        # Save; date_format controls how timestamps appear in the CSV text, dtype stays datetime in-memory
        out_csv = rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv'
        df_windows_total.to_csv(out_csv, index=False, date_format='%Y-%m-%d %H:%M:%S')
        print(f"saved → {out_csv}")
        
        ################################################################################
        #################################################################################
        #RD
        # --- Parse datetimes (keep as datetime64[ns], do NOT strftime) ---
        df_pot_rf['time']          = pd.to_datetime(df_pot_rf[time_pot_rf], errors='coerce')
        df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr[time_ntr], errors='coerce')
        df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf[time_rf],  errors='coerce')
        df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd[time_rd],  errors='coerce')
        
        # --- Make sure all 'time' columns are datetime ---
        # POT RD triggers
        if 'Time' in df_pot_rd.columns and 'time' not in df_pot_rd.columns:
            df_pot_rd = df_pot_rd.rename(columns={'Time': 'time'})
        
        df_pot_rd['time'] = pd.to_datetime(df_pot_rd[time_pot_rd], errors='coerce')
        
        
        def expand_daily_to_hourly(df_daily, time_col='time', value_col= rd_col):
            """
            For each daily timestamp in df_daily[time_col], create 24 hourly rows from
            00:00 to 23:00 with the same value in value_col.
            """
            df_daily = df_daily.dropna(subset=[time_col, value_col]).copy()
            df_daily[time_col] = pd.to_datetime(df_daily[time_col], errors='coerce')
            df_daily[value_col] = pd.to_numeric(df_daily[value_col], errors='coerce')
            df_daily = df_daily.dropna(subset=[time_col, value_col])
        
            pieces = []
            for t, v in zip(df_daily[time_col], df_daily[value_col]):
                d = pd.Timestamp(t).normalize()  # midnight of that day
                hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
                pieces.append(pd.DataFrame({time_col: hours, value_col: v}))
            if not pieces:
                return df_daily.iloc[0:0][[time_col, value_col]]  # empty with same cols
            return pd.concat(pieces, ignore_index=True)
        
        
        
        # Optional: drop rows with bad times
        df_pot_rd = df_pot_rd.dropna(subset=['time'])
        
        rd_windows = []
        
        for _, row in df_pot_rd.iterrows():
            pot_time = row['time']
            window_start = pot_time - pd.Timedelta(days=10)
            window_end   = pot_time + pd.Timedelta(days=10)
        
            window_df_rd_daily = df_timeseries_rd.loc[
                (df_timeseries_rd['time'] >= window_start) &
                (df_timeseries_rd['time'] <= window_end)
            ].copy()
        
            # ⬇️ expand daily RD to hourly within the window
            window_df_rd = expand_daily_to_hourly(
                window_df_rd_daily, time_col='time', value_col = rd_col
            )
        
            # keep NTR/RF hourly (unchanged)
            window_df_ntr = df_timeseries_ntr.loc[
                (df_timeseries_ntr['time'] >= window_start) &
                (df_timeseries_ntr['time'] <= window_end)
            ].copy()
        
            window_df_rf = df_timeseries_rf.loc[
                (df_timeseries_rf['time'] >= window_start) &
                (df_timeseries_rf['time'] <= window_end)
            ].copy()
        
            # inner-join on hourly timestamps; RD is now hourly by repetition
            window_df = (
                window_df_rd
                .merge(window_df_ntr, on='time', how='inner')
                .merge(window_df_rf, on='time', how='inner')
                .sort_values('time')
                .reset_index(drop=True)
            )
            window_df['event_time'] = pot_time
        
            if not window_df.empty:
                window_start = pot_time - pd.Timedelta(days=5)
                window_end   = pot_time + pd.Timedelta(days=5)
                sub = window_df.loc[
                window_df['time'].between(window_start, window_end),
                    ['time', acc_hr]
                ].dropna(subset=[acc_hr])
                
                if not sub.empty:
                    i = sub[acc_hr].idxmax()
                    max_rf_time = sub.at[i, 'time']
                    max_rf_val  = sub.at[i, acc_hr]
                    window_df['rf_lag_hours'] = (max_rf_time - pot_time) / pd.Timedelta(hours=1)
            
                sub = window_df.loc[
                window_df['time'].between(window_start, window_end),
                    ['time', ntr_col]
                ].dropna(subset=[ntr_col])
                
                if not sub.empty:
                    i = sub[ntr_col].idxmax()
                    max_ntr_time = sub.at[i, 'time']
                    max_ntr_val  = sub.at[i, ntr_col]
                    window_df['ntr_lag_hours'] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
                    
                else:
                    window_df['rf_lag_hours'] = np.nan
                    window_df['ntr_lag_hours']  = np.nan
                    
        
            rd_windows.append(window_df)
        
        df_windows_total = pd.concat(rd_windows, ignore_index=True)
        
        out_path = rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv'
        df_windows_total.to_csv(out_path, index=False)
        print(f"saved → {out_path}")

## Finding Location Based RD for Ocean Springs

In [11]:
import os
import pandas as pd
import numpy as np

In [ ]:
properties = ['Stationary','TimeVarying']
target_event = 10

core_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/POT_24hr_Jan26/Trivariate/OceanSprings/Reanalysis')
dataset_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/Datasets/Reanalysis')

time_rf = 'time'
time_ntr = 'time'       
time_rd = 'Time'
time_rd2 = 'Time' #if we have different rivers

time_pot_rf = 'time'
time_pot_ntr = 'time'       
time_pot_rd = 'time'

ntr_col = 'Average'
rd_col = 'RD_hourly'
rd_col2 = 'RD_hourly' #if we have different rivers
acc_hr = 'acc_24hr'

In [ ]:
for prop in properties:
    base_path = core_directory / prop

    df_timeseries_ntr = pd.read_csv(rf'{dataset_directory}/Compare_Points_OceanSprings_Avg.csv')
    df_timeseries_ntr = df_timeseries_ntr[[time_ntr, ntr_col]]
    
    df_timeseries_rf = pd.read_csv(rf'{dataset_directory}/APCP_Basin_Average_OceanSprings_Combined_accumulated.csv')
    df_timeseries_rf = df_timeseries_rf[[time_rf, acc_hr]]
    
    ############################################ Pascagoula #############################################
    df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/GloFAS_Pascagoula_Hourly.csv')
    # df_timeseries_rd = df_timeseries_rd.rename(columns={'valid_time':'time'})
    df_timeseries_rd = df_timeseries_rd[[time_rd, rd_col]]
    # df_timeseries_rd = df_timeseries_rd.rename(columns={'dis24':'dis24_hourly'})
    df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd[time_rd])        # ensure it's datetime
    
    
    
    df_pot_ntr_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv')
    df_pot_rf_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv')
    df_pot_rd_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv')
    df_pot_ntr_timeseries['time']  = pd.to_datetime(df_pot_ntr_timeseries['time'], errors='coerce')
    df_pot_rf_timeseries['time']   = pd.to_datetime(df_pot_rf_timeseries['time'],  errors='coerce')
    df_pot_rd_timeseries['time']   = pd.to_datetime(df_pot_rd_timeseries['time'],  errors='coerce')
    
    df_pot_ntr_timeseries_merged = pd.merge(df_pot_ntr_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    
    rd_start = df_timeseries_rd['time'].min()
    mask = (df_pot_ntr_timeseries_merged['time'] < rd_start) | (
        (df_pot_ntr_timeseries_merged['time'] >= rd_start) &
        df_pot_ntr_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_ntr_timeseries_merged = df_pot_ntr_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_ntr_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD_Pascagoula.csv', index=False)
    
    df_pot_rf_timeseries_merged = pd.merge(df_pot_rf_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rf_timeseries_merged['time'] < rd_start) | (
        (df_pot_rf_timeseries_merged['time'] >= rd_start) &
        df_pot_rf_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rf_timeseries_merged = df_pot_rf_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rf_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD_Pascagoula.csv', index=False)
    
    df_pot_rd_timeseries_merged = pd.merge(df_pot_rd_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rd_timeseries_merged['time'] < rd_start) | (
        (df_pot_rd_timeseries_merged['time'] >= rd_start) &
        df_pot_rd_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rd_timeseries_merged = df_pot_rd_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rd_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF_Pascagoula.csv', index=False)
    

    ############################################ Biloxi #############################################
    df_timeseries_rd = pd.read_csv(rf'{dataset_directory}/GloFAS_Biloxi_Hourly.csv')
    # df_timeseries_rd = df_timeseries_rd.rename(columns={'valid_time':'time'})
    df_timeseries_rd = df_timeseries_rd[[time_rd2, rd_col2]]
    # df_timeseries_rd = df_timeseries_rd.rename(columns={'dis24':'dis24_hourly'})
    df_timeseries_rd['time'] = pd.to_datetime(df_timeseries_rd[time_rd2])        # ensure it's datetime

    df_pot_ntr_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD.csv')
    df_pot_rf_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD.csv')
    df_pot_rd_timeseries = pd.read_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF.csv')
    df_pot_ntr_timeseries['time']  = pd.to_datetime(df_pot_ntr_timeseries['time'], errors='coerce')
    df_pot_rf_timeseries['time']   = pd.to_datetime(df_pot_rf_timeseries['time'],  errors='coerce')
    df_pot_rd_timeseries['time']   = pd.to_datetime(df_pot_rd_timeseries['time'],  errors='coerce')
    
    df_pot_ntr_timeseries_merged = pd.merge(df_pot_ntr_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    
    rd_start = df_timeseries_rd['time'].min()
    mask = (df_pot_ntr_timeseries_merged['time'] < rd_start) | (
        (df_pot_ntr_timeseries_merged['time'] >= rd_start) &
        df_pot_ntr_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_ntr_timeseries_merged = df_pot_ntr_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_ntr_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_NTR_Associated_RF_RD_Biloxi.csv', index=False)
    
    df_pot_rf_timeseries_merged = pd.merge(df_pot_rf_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rf_timeseries_merged['time'] < rd_start) | (
        (df_pot_rf_timeseries_merged['time'] >= rd_start) &
        df_pot_rf_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rf_timeseries_merged = df_pot_rf_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rf_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RF_Associated_NTR_RD_Biloxi.csv', index=False)
    
    df_pot_rd_timeseries_merged = pd.merge(df_pot_rd_timeseries,df_timeseries_rd, on ='time', how = 'outer')
    mask = (df_pot_rd_timeseries_merged['time'] < rd_start) | (
        (df_pot_rd_timeseries_merged['time'] >= rd_start) &
        df_pot_rd_timeseries_merged.notna().all(axis=1)   # keep only fully populated rows after rd_start
    )
    df_pot_rd_timeseries_merged = df_pot_rd_timeseries_merged.loc[mask].reset_index(drop=True)
    df_pot_rd_timeseries_merged.to_csv(rf'{base_path}/TimeSeries_POT_RD_Associated_NTR_RF_Biloxi.csv', index=False)